# Projet - Tache 1 : Collecte de donnees

Version simple :
- requete SPARQL avec `SPARQLWrapper`
- resultats dans un `DataFrame` pandas
- une petite fonction pour lire les metadonnees Commons
- telechargement des vignettes et export vers `data/images_metadata.json`


In [ ]:
# A executer une fois si besoin
# !pip install SPARQLWrapper pandas Pillow requests

import html
import json
import re
import time
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
import requests
from PIL import Image
from PIL.ExifTags import TAGS
from SPARQLWrapper import JSON, SPARQLWrapper

BASE_DIR = Path('.')
IMAGES_DIR = BASE_DIR / 'images'
DATA_DIR = BASE_DIR / 'data'
IMAGES_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

USER_AGENT = 'CPE-ML-Project/1.0'


In [ ]:
query = '''
SELECT ?fleur ?fleurLabel ?image
       (GROUP_CONCAT(DISTINCT ?couleurLabel; separator=", ") AS ?couleurs)
WHERE {
  ?fleur wdt:P2827 ?couleur ;
         wdt:P18 ?image .
  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "fr,en".
  }
}
GROUP BY ?fleur ?fleurLabel ?image
LIMIT 120
'''

sparql = SPARQLWrapper('https://query.wikidata.org/sparql', agent=USER_AGENT)
sparql.setQuery(query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()['results']['bindings']

df = pd.DataFrame([
    {
        'wikidata_id': r['fleur']['value'].split('/')[-1],
        'label': r.get('fleurLabel', {}).get('value', ''),
        'image_url': r['image']['value'],
        'colors': r.get('couleurs', {}).get('value', '')
    }
    for r in results
]).drop_duplicates(subset='image_url').reset_index(drop=True)

print('Nombre de lignes :', len(df))
df.head()


In [ ]:
def clean_html(value):
    if value is None:
        return None
    value = html.unescape(str(value))
    value = re.sub(r'<[^>]+>', ' ', value)
    return re.sub(r'\s+', ' ', value).strip() or None


def get_commons_metadata(image_url):
    commons_title = 'File:' + unquote(image_url.split('/Special:FilePath/')[1])
    response = requests.get(
        'https://commons.wikimedia.org/w/api.php',
        params={
            'action': 'query',
            'titles': commons_title,
            'prop': 'imageinfo',
            'iiprop': 'url|size|mime|extmetadata',
            'iiurlwidth': 800,
            'format': 'json'
        },
        headers={'User-Agent': USER_AGENT},
        timeout=60
    )
    info = next(iter(response.json()['query']['pages'].values()))['imageinfo'][0]
    ext = info.get('extmetadata', {})
    return {
        'commons_title': commons_title,
        'download_url': info.get('thumburl') or info.get('url'),
        'source_page_url': info.get('descriptionurl'),
        'source_file_url': info.get('url'),
        'license_short_name': clean_html(ext.get('LicenseShortName', {}).get('value')),
        'license_url': clean_html(ext.get('LicenseUrl', {}).get('value')),
        'artist': clean_html(ext.get('Artist', {}).get('value')),
        'credit': clean_html(ext.get('Credit', {}).get('value')),
        'date_time_original': clean_html(ext.get('DateTimeOriginal', {}).get('value')),
        'original_file_size_kb': round((info.get('size') or 0) / 1024, 2),
        'original_width': info.get('width'),
        'original_height': info.get('height')
    }


def get_exif(image_path):
    keep = {'Make', 'Model', 'DateTime', 'DateTimeOriginal'}
    with Image.open(image_path) as img:
        return {
            TAGS.get(tag_id, tag_id): str(value)
            for tag_id, value in img.getexif().items()
            if TAGS.get(tag_id, tag_id) in keep
        }


In [ ]:
metadata = []

for i, row in df.head(100).iterrows():
    try:
        commons = get_commons_metadata(row['image_url'])
        extension = Path(commons['download_url']).suffix.lower() or '.jpg'
        file_name = f'image_{i+1:03d}{extension}'
        image_path = IMAGES_DIR / file_name

        if not image_path.exists():
            response = requests.get(commons['download_url'], headers={'User-Agent': USER_AGENT}, timeout=60)
            response.raise_for_status()
            image_path.write_bytes(response.content)

        with Image.open(image_path) as img:
            width, height = img.size
            image_format = img.format

        label = row['label'] if row['label'] and not row['label'].startswith('Q') else commons['commons_title'].replace('File:', '')

        metadata.append({
            'file_name': file_name,
            'wikidata_id': row['wikidata_id'],
            'label': label,
            'colors': row['colors'],
            'width': width,
            'height': height,
            'format': image_format,
            'file_size_kb': round(image_path.stat().st_size / 1024, 2),
            'source_url': row['image_url'],
            'source_page_url': commons['source_page_url'],
            'source_file_url': commons['source_file_url'],
            'license_short_name': commons['license_short_name'],
            'license_url': commons['license_url'],
            'artist': commons['artist'],
            'credit': commons['credit'],
            'date_time_original': commons['date_time_original'],
            'original_file_size_kb': commons['original_file_size_kb'],
            'original_width': commons['original_width'],
            'original_height': commons['original_height'],
            'exif': get_exif(image_path)
        })

        print(f'{len(metadata):03d} - {file_name}')
        time.sleep(0.1)

    except Exception as e:
        print('Erreur :', row['image_url'], e)

metadata_df = pd.DataFrame(metadata)
metadata_df.head()


In [ ]:
output_path = DATA_DIR / 'images_metadata.json'
metadata_df.to_json(output_path, orient='records', force_ascii=False, indent=2)

print('JSON sauvegarde dans :', output_path)
print('Nombre final :', len(metadata_df))
metadata_df.head()
